# FinSight Phase 3 — Deep Dive: Agents & Orchestration (LangGraph)

**Who this is for:** you've seen Phases 1–2 (retrieve → cited answer, measured
at 86%). Phase 3 turns that single pipeline into a **team of specialist agents**
that plan, fetch real numbers, cross-check each other, and pause for a human
when unsure.

### The one-sentence shift

> Phases 1–2 were a **fixed pipeline** — same steps every time. Phase 3 is an
> **agent system** — a supervisor decides *at runtime* which specialists to call.

### CV analogies (to anchor the new words)

| You know (CV / ML) | Phase 3 concept |
|---|---|
| A fixed forward pass | the Phase 2 pipeline |
| A **computation graph** with branching | **LangGraph** — nodes + a shared state dict |
| An **ensemble** of specialist models | supervisor routing to RAG + quant agents |
| A **two-stage detector** (propose → verify) | synthesis writes, **critic verifies** |
| Reading a value from a **lookup table**, not predicting it | **quant agent** fetching numbers from SEC XBRL |

### How we'll learn it

The real system is a LangGraph graph. But to *understand* it, we'll thread the
same **state dict** through each agent **by hand**, one at a time, watching what
each one adds. Then we'll run the assembled graph and see the identical result.

```
state = {question}
 → supervisor  adds  {plan, sub_tasks, route}
 → rag agent   adds  {evidence}
 → quant agent adds  {figures}
 → synthesis   adds  {draft}
 → critic      adds  {verdict}   → accept | retry | human
```

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option("display.max_colwidth", 100)
from IPython.display import Markdown, display
print("ready — note: agent cells call gpt-5-mini and take a few seconds each")

ready — note: agent cells call gpt-5-mini and take a few seconds each


# Part 1 — The quant agent's foundation: real numbers, never the model's

The single most important rule in financial AI: **figures never come from the
LLM's memory.** Ask any model for JPMorgan's 2024 net interest income and it
will give you a confident, wrong number.

So Phase 3 adds a **quant agent** that fetches figures from the SEC's free
**XBRL company-facts API** — the same filings' machine-readable numbers. Before
the agent, let's look at the raw tools it will call (`tools/xbrl.py`). These are
plain functions — no LLM yet.

In [2]:
from tools import xbrl

# One reported figure, with provenance
jpm_assets = xbrl.get_metric("JPM", "total_assets", 2024)
print(json.dumps(jpm_assets, indent=2))

{
  "ticker": "JPM",
  "metric": "total_assets",
  "concept": "Assets",
  "year": 2024,
  "value": 4002814000000.0,
  "unit": "USD",
  "source": "SEC XBRL Assets CY2024"
}


Notice the result carries its **source** (`SEC XBRL Assets CY2024`) — the
numeric equivalent of a citation. That string travels with the figure all the
way to the final answer.

A trend, and a derived bank metric:

In [3]:
trend = xbrl.compare_trend("JPM", "net_interest_income", [2023, 2024, 2025])
print("JPM net interest income:")
for y, v in trend["points"].items():
    print(f"   {y}: ${v/1e9:.1f}B")
print("   first->last change:", trend["pct_change_first_to_last"], "%")
print("   source:", trend["source"])

JPM net interest income:
   2023: $89.3B
   2024: $92.6B
   2025: $95.4B
   first->last change: 6.9 %
   source: SEC XBRL InterestIncomeExpenseNet


### Why this was the hard part (a real lesson)

The SEC API is messier than it sounds. Two problems the tools had to solve:

1. **Issuers switch tags.** JPMorgan used to report interest income under the
   US-GAAP concept `InterestAndDividendIncomeOperating`, then switched to
   `InterestIncomeOperating`. Net interest income is reported *directly* as
   `InterestIncomeExpenseNet`. So a friendly name like `"net_interest_income"`
   maps to a *list* of candidate tags, and we pick the one that actually has
   data for the requested years.
2. **The `frame` field is unreliable** for recent filings, so annual figures
   are keyed by the period-END year, selecting `form=10-K, fp=FY`, keeping the
   latest-*filed* value when a later 10-K restates a prior year.

A bad metric fails cleanly (no guessing) — which is exactly what an
anti-hallucination system must do:

In [4]:
print(json.dumps(xbrl.get_metric("JPM", "unicorn_count", 2024), indent=2))

{
  "ticker": "JPM",
  "metric": "unicorn_count",
  "year": 2024,
  "value": null,
  "error": "no reported value for unicorn_count 2024",
  "available_years": []
}


# Part 2 — Tool calling: how the LLM *uses* those functions

An **agent** = an LLM given tools + a loop. We describe each XBRL function to
the model as a callable "tool" (name, description, parameter schema). The model
then emits a structured request like `get_metric(ticker="JPM", metric="net_interest_income", year=2024)`;
our code runs it and feeds the result back; the model repeats until it has what
it needs.

<span style="color:#3b7fd0;font-weight:600">The model decides *which* numbers to
fetch; the API decides *what the values are*.</span>

Let's run the quant agent node on two banks and watch it fan out into tool
calls.

In [5]:
from agents.quant_agent import quant_node

state = {
    "question": "Compare net interest income for JPMorgan and Bank of America, 2023 vs 2024",
    "sub_tasks": [
        {"kind": "quant", "ticker": "JPM", "detail": "net interest income 2023 and 2024"},
        {"kind": "quant", "ticker": "BAC", "detail": "net interest income 2023 and 2024"},
    ],
}
out = quant_node(state)
print(out["trace"][-1], "\n")
display(pd.DataFrame(out["figures"])[["ticker", "metric", "year", "value", "source"]])

[quant]      4 tool call(s) -> 4 figure(s) from SEC XBRL 



,ticker,metric,year,value,source
0,JPM,net_interest_income,2023,8.926700e+10,SEC XBRL InterestIncomeExpenseNet CY2023
1,JPM,net_interest_income,2024,9.258300e+10,SEC XBRL InterestIncomeExpenseNet CY2024
2,BAC,net_interest_income,2023,5.693100e+10,SEC XBRL InterestIncomeExpenseNet CY2023
3,BAC,net_interest_income,2024,5.606000e+10,SEC XBRL InterestIncomeExpenseNet CY2024


Every row came from a tool call against `data.sec.gov`. The LLM never typed
a number — it only chose which tool to call. That is the whole point.

# Part 3 — The supervisor: plan and route

The **supervisor** (a cheap model) reads the question and decides: which
companies, and does this need *text* (rag), *numbers* (quant), or both? Simple
single-company text questions route straight to RAG so they never pay for the
quant path — **dynamic routing = cost control**.

Watch it decompose two different questions.

In [6]:
from agents.supervisor import supervisor_node

simple = supervisor_node({"question": "What cybersecurity risks did JPMorgan flag?"})
print("SIMPLE question ->", simple["route"])
for t in simple["sub_tasks"]:
    print("   ", t["kind"], t.get("ticker"), "-", t["detail"][:60])

SIMPLE question -> rag_only
    rag JPM - Fetch JPMorgan's most recent 10-K (and latest 10-Q if applic


In [7]:
complex_q = supervisor_node({"question":
    "Compare the net interest income trend for JPMorgan vs Bank of America "
    "since 2023, and what risks did management flag?"})
print("COMPARISON question ->", complex_q["route"])
for t in complex_q["sub_tasks"]:
    print("   ", t["kind"], t.get("ticker"), "-", t["detail"][:60])

COMPARISON question -> fanout
    quant JPM - Fetch reported net interest income (NII) amounts for JPMorga
    rag JPM - Retrieve management's qualitative discussion and risk disclo
    quant BAC - Fetch reported net interest income (NII) amounts for Bank of
    rag BAC - Retrieve management's qualitative discussion and risk disclo


See the difference: the simple question produced **rag-only** (quant never
runs), while the comparison fanned out into per-company **rag + quant** tasks.
That per-company decomposition is exactly what fixes the Phase 2 weak spot —
multi-company comparisons that one retrieval pass couldn't handle.

# Part 4 — The RAG agent (Phase 2, now a node)

No new retrieval logic — the RAG node simply runs each `rag` sub-task through
the **measured Phase 2 winner** (`rewrite+hybrid+rerank`, 86% hit rate) and
appends the cited chunks to `state["evidence"]`. Reuse, not reinvention.

In [8]:
from agents.rag_agent import rag_node

rag_out = rag_node({"sub_tasks": [
    {"kind": "rag", "ticker": "JPM", "detail": "management-flagged risks in risk factors"}]})
print(rag_out["trace"][-1])
display(pd.DataFrame([{"citation": e["citation"], "item": e["item"],
                       "preview": e["text"][40:130].replace(chr(10), " ")}
                      for e in rag_out["evidence"]]))

C:\Users\subha\OneDrive\Desktop\Subham\finsight\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6867.73it/s]

[rag]        JPM: "management-flagged risks in risk factors" -> 6 chunks


,citation,item,preview
0,"[JPM 10-K 2025, Item 15]",15,"nd Financial Statement Schedules (incl. content incorporated by reference: MD&A, financial"
1,"[JPM 10-K 2025, Item 1A]",1A,rs] Item 1A. Risk Factors. The following discussion sets forth the material risk factors
2,"[JPM 10-K 2025, Item 1A]",1A,rs] Part I and to maintain an effective risk management framework is highly dependent on i


# Part 5 — Synthesis: merge text + numbers into a cited answer

The synthesis agent reuses the Phase 1 **cite-or-refuse** discipline, extended
so numeric claims cite their XBRL source too. We give it the evidence and
figures we gathered above.

In [9]:
from agents.synthesis import synthesis_node

state = {
    "question": "Compare JPMorgan and Bank of America net interest income (2023-2024) and JPM's risks.",
    "evidence": rag_out["evidence"],
    "figures": out["figures"],   # from Part 2
}
syn = synthesis_node(state)
display(Markdown(syn["draft"]))

Net interest income — reported levels
- JPMorgan Chase: $89,267,000,000 in 2023 [SEC XBRL InterestIncomeExpenseNet CY2023].  
- JPMorgan Chase: $92,583,000,000 in 2024 [SEC XBRL InterestIncomeExpenseNet CY2024].  
- Bank of America: $56,931,000,000 in 2023 [SEC XBRL InterestIncomeExpenseNet CY2023].  
- Bank of America: $56,060,000,000 in 2024 [SEC XBRL InterestIncomeExpenseNet CY2024].  

Comparison
- JPMorgan’s reported net interest income was higher than Bank of America’s in both 2023 and 2024 (2023: $89,267,000,000 v. $56,931,000,000; 2024: $92,583,000,000 v. $56,060,000,000) [SEC XBRL InterestIncomeExpenseNet CY2023; SEC XBRL InterestIncomeExpenseNet CY2024].  

JPMorgan — principal risks (summary)
- The company identifies principal risk factors that include legal and regulatory, political, market and credit risks [JPM 10-K 2025, Item 1A].  
- JPMorgan describes a firmwide risk management objective and governance framework intended to balance serving stakeholders and protecting the firm’s safety and soundness [JPM 10-K 2025, Item 15].  
- Operational risks tied to staffing and workforce issues (including staffing shortages, employee failure to adhere to controls, inability to work effectively, theft/fraud or human error) are noted as potential material adverse effects on the business [JPM 10-K 2025, Item 1A].  
- The firm highlights substantial legal and operational risks related to processing and safeguarding personal information, including compliance costs, cross‑border data restrictions, conflicting laws and potential enforcement or litigation if data is mishandled [JPM 10-K 2025, Item 1A].  

Insufficient evidence in the indexed filings.
- No other financial metrics (e.g., net interest margin, dollar changes, or percentages) for 2023–2024 are provided in the indexed figures to support further quantitative comparison beyond the reported net interest income totals.

# Part 6 — The critic, and the bug that taught me the most

The **critic** is the control layer: it re-reads the draft and checks that every
claim maps to a passage or a fetched figure. It can **accept**, send back for a
**bounded retry**, or escalate to a **human**.

### The debugging story (worth its own section)

When I first built the critic it flagged ~10 "unsupported claim" issues on a
*perfectly good* answer and never accepted it — every run looped to the
human-review gate. The answer wasn't wrong. So what happened?

**The critic saw less evidence than the writer did.** Synthesis showed the LLM
1,200 characters per passage; the critic showed it only **300**. The supporting
sentences were simply truncated out of the critic's view — it was judging a
smaller excerpt, not the answer. Let's reproduce it.

In [10]:
from config import CHAT_DEPLOYMENT
from ingestion.embedder import get_openai
from agents.critic import SYSTEM as CRITIC_SYSTEM
import re, json as _json

client = get_openai()

# A CONTROLLED experiment: one passage whose key fact ("ransomware") sits
# deliberately PAST character 300, and a draft claim that relies on it.
passage = (
    "[JPM | 10-K FY2025 | Item 1A: Risk Factors]\n"
    "JPMorganChase faces a broad range of operational risks arising from its "
    "reliance on complex technology systems, third-party vendors, and global "
    "operations across many jurisdictions, any of which could be disrupted by "
    "errors, external events, or malicious activity affecting the Firm. "               # ~300 chars so far
    "In particular, cyber attackers may attempt to disrupt or degrade the Firm's "
    "services, steal money, or extort money from JPMorganChase or its clients "
    "through ransomware."
)
draft = ("JPMorgan warns that cyber attackers may extort money from the Firm or "
         "its clients through ransomware [JPM 10-K 2025, Item 1A].")

def run_critic(passage_text, per_passage_chars):
    ev = f"- [JPM 10-K 2025, Item 1A]: {passage_text[:per_passage_chars]}"
    user = f"QUESTION: What ransomware risk did JPMorgan flag?\n\nPASSAGES:\n{ev}\n\nFIGURES:\n(none)\n\nDRAFT:\n{draft}"
    raw = client.chat.completions.create(model=CHAT_DEPLOYMENT, messages=[
        {"role": "system", "content": CRITIC_SYSTEM},
        {"role": "user", "content": user}]).choices[0].message.content
    return _json.loads(re.search(r"\{.*\}", raw, re.S).group(0))

print("The word 'ransomware' first appears at character",
      passage.find("ransomware"), "of the passage.\n")
tiny = run_critic(passage, 300)     # the bug: support truncated away
full = run_critic(passage, 2500)    # the fix: critic sees the whole passage
print(f"critic shown 300 chars  (support hidden) : ok={tiny['ok']}  issues={len(tiny['issues'])}")
print(f"critic shown 2500 chars (support visible): ok={full['ok']}  issues={len(full['issues'])}")

The word 'ransomware' first appears at character 485 of the passage.



critic shown 300 chars  (support hidden) : ok=False  issues=1
critic shown 2500 chars (support visible): ok=True  issues=0


Same draft, same critic prompt — the supporting sentence sits at character
~490, so the 300-char view **cannot see it** and the critic rejects a correct
claim; the full view sees it and accepts. Only the *amount of evidence shown*
changed, and the verdict flips.

The real fix was two lines: show the critic the full passage text (2,500 chars,
matching synthesis) and instruct it to accept *faithful paraphrase* rather than
demand verbatim matches.

**The lesson (a keeper for interviews):** *a verifier is only as good as the
evidence you show it.* An AI reviewer that can't see the source will invent
objections — the failure looks like a bad model but is really a plumbing bug.

# Part 7 — Assembling the graph

Everything above ran by hand. LangGraph wires the same nodes into a graph with a
shared state, conditional edges, a **bounded retry loop**, and a
**human-in-the-loop interrupt**.

```
supervisor → rag → quant → synthesis → critic ─┬─ accept → done
                              ▲                 ├─ retry  → revise ─┐ (max 2)
                              └─────────────────┘                  │
                              (revise loops back to synthesis) ◀───┘
                                                └─ human → interrupt (pause)
```

The `route_after_critic` function is the branch: accept if confident, else retry
while the budget lasts, else escalate to a human.

In [11]:
from agents.critic import route_after_critic, MAX_RETRIES
import inspect
print("bounded retry budget:", MAX_RETRIES)
print(inspect.getsource(route_after_critic))

bounded retry budget: 2
def route_after_critic(state: FinSightState) -> str:
    """Conditional edge: accept, retry synthesis, or escalate to a human."""
    verdict = state.get("verdict", {})
    if verdict.get("ok") and verdict.get("confidence", 0) >= 0.6:
        return "accept"
    if state.get("retries", 0) < MAX_RETRIES:
        return "retry"
    return "human"  # exhausted retries and still not confident



# Part 8 — Run the whole graph (the deliverable)

Now the assembled system on the full comparison question. We use an in-memory
checkpointer here for speed; the `agents.run` CLI uses the **Postgres**
checkpointer (Part 9). This cell makes several LLM calls across the agents, so
give it a minute or two.

In [12]:
from langgraph.checkpoint.memory import MemorySaver
from agents.graph import build_graph

app = build_graph(checkpointer=MemorySaver())
cfg = {"configurable": {"thread_id": "notebook-demo"}}
result = app.invoke({"question":
    "Compare the net interest income trend for JPMorgan vs Bank of America "
    "since 2023, and what risks did management flag?", "trace": []}, cfg)

print("=== AGENT-HOP TRACE ===")
for line in result["trace"]:
    print(" ", line)

=== AGENT-HOP TRACE ===
  [supervisor] plan: 2 rag + 2 quant sub-task(s) -> route=fanout
  [rag]        JPM: "Retrieve management discussion and flagged risks fro" -> 6 chunks
  [rag]        BAC: "Retrieve management discussion and flagged risks fro" -> 6 chunks
  [quant]      8 tool call(s) -> 6 figure(s) from SEC XBRL
  [synthesis]  draft written (9 passages, 6 figures)
  [critic]     ok, confidence=0.92
  [done]       answer accepted


In [13]:
display(Markdown("### Final answer\n\n" + (result.get("answer") or "(none)")))

### Final answer

JPMorgan trend — JPMorgan’s net interest income rose from $89,267,000,000 in 2023 [SEC XBRL InterestIncomeExpenseNet CY2023] to $92,583,000,000 in 2024 [SEC XBRL InterestIncomeExpenseNet CY2024] and to $95,443,000,000 in 2025 [SEC XBRL InterestIncomeExpenseNet CY2025] [JPM 10-K 2025, Item 15].

Bank of America trend — Bank of America’s net interest income was $56,931,000,000 in 2023 [SEC XBRL InterestIncomeExpenseNet CY2023], decreased to $56,060,000,000 in 2024 [SEC XBRL InterestIncomeExpenseNet CY2024], then increased to $60,096,000,000 in 2025 [SEC XBRL InterestIncomeExpenseNet CY2025] [BAC 10-K 2025, Item 7].

Comparison — JPMorgan shows a steady year‑over‑year increase in net interest income across 2023–2025, whereas Bank of America experienced a slight decline in 2024 followed by a rebound in 2025 [JPM 10-K 2025, Item 15; BAC 10-K 2025, Item 7].

Risks JPMorgan flagged — Management highlighted risks from counterparties being unable to post collateral or disputes over collateral valuation, concentrations of credit and market risk (which could require higher allowances), potential losses in investment‑portfolio and market‑making activities, and sensitivity of consumer businesses to adverse economic conditions including prolonged or significant changes in interest rates and collateral values [JPM 10-K 2025, Item 1A].

Risks Bank of America flagged — Bank of America identified seven principal risk types (strategic, credit, market — including interest‑rate risk, liquidity, compliance, operational and reputational), and emphasized conduct risk, stress testing, and governance/monitoring processes as central to managing those risks [BAC 10-K 2025, Item 1A].

Read the trace: the supervisor planned 2 rag + 2 quant sub-tasks, the quant
agent made real XBRL tool calls, synthesis merged everything, and the critic
**accepted** it. Every number in the answer is tagged with its `SEC XBRL`
source; every qualitative claim carries its 10-K citation.

# Part 9 — Human-in-the-loop & the Postgres checkpointer

Two governance features finance demands:

- **Checkpointer** — LangGraph saves state after *every* node into Postgres (the
  same `finsight-db`). A run is resumable and fully **auditable**: you can replay
  exactly what each agent saw. The `agents.run` CLI already wrote real
  checkpoints — let's count them.
- **Human-in-the-loop** — when the critic stays unhappy after the retry budget,
  the graph calls `interrupt()`, which **pauses** the run and persists its state.
  A person approves/edits/rejects, and the graph **resumes** from the saved
  checkpoint. (Verified separately; hard to trigger on good data because the
  answers keep passing.)

In [14]:
import psycopg
from config import DATABASE_URL
with psycopg.connect(DATABASE_URL) as conn:
    tbls = [r[0] for r in conn.execute(
        "SELECT tablename FROM pg_tables WHERE tablename LIKE 'checkpoint%'").fetchall()]
    n = conn.execute("SELECT count(*) FROM checkpoints").fetchone()[0]
print("LangGraph checkpoint tables in finsight-db:", tbls)
print("checkpoints persisted so far:", n, "(one row per node hop, per run)")

LangGraph checkpoint tables in finsight-db: ['checkpoint_blobs', 'checkpoint_migrations', 'checkpoint_writes', 'checkpoints']
checkpoints persisted so far: 8 (one row per node hop, per run)


# Recap — what Phase 3 added

| Piece | Role | The idea |
|---|---|---|
| Supervisor | plan + route | cheap model; text-only skips the quant path (cost control) |
| RAG agent | retrieve text | the Phase 2 winner (86%) as a node — reuse |
| Quant agent | fetch numbers | tool-calling over SEC XBRL; **numbers never from memory** |
| Synthesis | write | cite-or-refuse, now citing figure sources too |
| Critic | verify | every claim → a passage or a figure; bounded retry |
| Checkpointer | persist | Postgres save/resume → auditable + resumable |
| HITL interrupt | escalate | pause for a human on low confidence |

**What you now understand:** an *agent* is an LLM + tools + a loop; *LangGraph*
is a state machine wiring those agents together; the *quant agent* reads real
numbers instead of predicting them; and the *critic* is the control layer that
makes agent autonomy acceptable in finance — as long as you show it the
evidence.

**Next (Phase 4 — productionization):** wrap this graph in a streaming FastAPI
service, trace every node's tokens/latency/cost with Langfuse, add guardrails
(prompt-injection and PII scans), and containerize it — `docker compose up`.